# ktune · QLoRA fine-tuning with custom Triton kernels

Run this on a **GPU runtime** (Colab: Runtime → Change runtime type → T4 GPU).

It fine-tunes a small model with 4-bit QLoRA twice — once stock, once patched
with `ktune`'s fused kernels — and compares tokens/s and peak VRAM.

Read `docs/` in the repo alongside this for the *why* behind each kernel.

In [ ]:
# 1. Install ktune + the app extra (transformers, peft, bitsandbytes, datasets)
!git clone https://github.com/aabhimittal/llm-tuning-kernel.git
%cd llm-tuning-kernel
!pip install -q -e ".[gpu,app]"

In [ ]:
# 2. Sanity check: kernels match their references on this GPU
!pytest tests/ -m gpu -q

In [ ]:
# 3. See the memory win of Fused Linear Cross-Entropy directly
!python benchmarks/bench_flce.py --hardware t4

In [ ]:
# 4a. Baseline QLoRA fine-tune (no ktune)
!python examples/finetune_qlora.py --model Qwen/Qwen2.5-0.5B --steps 30

In [ ]:
# 4b. Same fine-tune, patched with ktune fused kernels
!python examples/finetune_qlora.py --model Qwen/Qwen2.5-0.5B --steps 30 --ktune

Compare the `throughput` and `peak VRAM` lines from 4a vs 4b. The loss curve
should be essentially identical — the kernels change *how* the math is computed,
not *what*. See `docs/08-applying-to-models.md`.